# 02. Inferência de três jeitos

Estimar o ATE é só metade do trabalho; a outra metade é dizer **quão certo** estamos. A biblioteca oferece três caminhos, com interpretações diferentes:

- **Randomização** (`RandomizationTest`): testa a hipótese nula forte de Fisher.
- **Neyman** (`NeymanCI`): variância de população finita.
- **Bootstrap** (`BootstrapCI`): variância de superpopulação.

Vamos rodar os três no **mesmo** experimento e comparar.

## O experimento

Mesmo padrão do notebook 01: potential outcomes com efeito verdadeiro `0.5`, randomização completa, outcome observado.

In [ ]:
import numpy as np
import pandas as pd

from skxperiments.core.assignment import CRDAssignment
from skxperiments.design.crd import CRD
from skxperiments.estimators.difference_in_means import DifferenceInMeans

rng = np.random.default_rng(42)
n = 200
y0 = rng.normal(size=n)
tau = 0.5
y1 = y0 + tau

df = pd.DataFrame({"x": rng.normal(size=n)})
design = CRD(p=0.5, seed=42)
assignment = design.randomize(df)
t = assignment.data_[assignment.treatment_col_].to_numpy()
data = assignment.data_.copy()
data["y"] = np.where(t == 1, y1, y0)
assignment = CRDAssignment(
    data=data, treatment_col=assignment.treatment_col_, design=design, seed=42
)

## 1. Randomização (sharp null de Fisher)

Permuta o tratamento milhares de vezes sob a nula de "nenhum efeito em nenhuma unidade". Dá um **p-valor**, mas não um erro-padrão nem um intervalo de confiança.

In [ ]:
from skxperiments.inference import RandomizationTest

res_rand = RandomizationTest(
    estimator=DifferenceInMeans(outcome_col="y"), n_permutations=500, seed=0
).fit(assignment).estimate()
print(f"Randomizacao: ATE={res_rand.ate:.3f}, p={res_rand.p_value:.4f}")

## 2. Neyman (população finita)

Variância conservadora do ATE para **estas** unidades. Dá erro-padrão, intervalo de confiança e p-valor de Wald.

In [ ]:
from skxperiments.inference import NeymanCI

res_ney = NeymanCI(
    estimator=DifferenceInMeans(outcome_col="y")
).fit(assignment).estimate()
print(
    f"Neyman: ATE={res_ney.ate:.3f}, SE={res_ney.se:.3f}, "
    f"IC=({res_ney.ci[0]:.3f}, {res_ney.ci[1]:.3f}), p={res_ney.p_value:.4f}"
)

## 3. Bootstrap (superpopulação)

Reamostra as unidades dentro de cada braço para aproximar a distribuição amostral, tratando-as como amostra de uma população maior.

In [ ]:
from skxperiments.inference import BootstrapCI

res_boot = BootstrapCI(
    estimator=DifferenceInMeans(outcome_col="y"), n_resamples=1000, seed=0
).fit(assignment).estimate()
print(
    f"Bootstrap: ATE={res_boot.ate:.3f}, SE={res_boot.se:.3f}, "
    f"IC=({res_boot.ci[0]:.3f}, {res_boot.ci[1]:.3f}), p={res_boot.p_value:.4f}"
)

## Comparando

Os três concordam no ATE (é o mesmo estimador); o que muda é a medida de incerteza. A randomização não entrega SE/IC nativamente (por isso `None` na tabela).

In [ ]:
pd.DataFrame(
    [
        {"metodo": "Randomizacao", "ATE": res_rand.ate, "SE": res_rand.se, "p": res_rand.p_value},
        {"metodo": "Neyman", "ATE": res_ney.ate, "SE": res_ney.se, "p": res_ney.p_value},
        {"metodo": "Bootstrap", "ATE": res_boot.ate, "SE": res_boot.se, "p": res_boot.p_value},
    ]
)

## Quando usar cada um

- **Randomização**: quando você quer um p-valor que dependa só do desenho, sem pressupostos distribucionais. Testa o sharp null.
- **Neyman**: quando a pergunta é sobre **estas** unidades (população finita) e você quer um IC rápido e conservador.
- **Bootstrap**: quando as unidades são amostra de uma população maior (superpopulação) e a pergunta é sobre ela.

**Próximo:** `03. Reduzir variância` mostra como covariáveis (Lin e CUPED) deixam o IC mais estreito.